<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%); padding:48px 40px; border-radius:16px; text-align:center; margin-bottom:8px;">
  <h1 style="color:#e94560; font-size:2.4em; font-weight:800; letter-spacing:2px; margin:0 0 12px 0; text-transform:uppercase;">Telecom Customer Churn Prediction</h1>
  <p style="color:#a8b2d8; font-size:1.15em; margin:0; letter-spacing:1px;">End-to-End Machine Learning Pipeline &nbsp;|&nbsp; 100,000 Customers &nbsp;|&nbsp; 100 Features &nbsp;|&nbsp; 11 Models</p>
  <hr style="border:none; border-top:1px solid #e9456033; margin:20px auto; width:60%;">
  <p style="color:#7f8ca8; font-size:0.9em; margin:0;">EDA → Cleaning → Feature Engineering → Modelling → Ensembles → Revenue at Risk → Conclusion</p>
</div>

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 1 &mdash; Business Understanding</h2>
  <p style="color:#444; line-height:1.7;">Customer churn costs 5–7× more per acquisition than retention. A 5% improvement in retention can lift profitability by 25–95%. This pipeline predicts, 30–60 days ahead, which customers are at high risk of churning — enabling proactive retention campaigns. <strong>Primary metric: Recall</strong> (missing a churner is far more costly than a false alarm). Secondary: F1, AUC, Precision.</p>
</div>

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 2 &mdash; Import Libraries</h2>
  <p style="color:#444; line-height:1.7;">All libraries imported upfront. <code>random_state=42</code> enforced globally via <code>SEED</code> for full reproducibility. Two model families require different data versions: scaled (LR, KNN) and unscaled (tree-based).</p>
</div>

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import json, os

# ── Core Data ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import missingno as msno
    MISSINGNO = True
except ImportError:
    MISSINGNO = False
    print("missingno not installed — missing-value matrix will use seaborn heatmap instead")

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif

# ── Imbalance ─────────────────────────────────────────────────────────────────
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn not installed — will fall back to class_weight='balanced'")

# ── Models ────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    XGB_AVAILABLE = False
    print("xgboost not installed — using GradientBoostingClassifier as substitute")

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, classification_report
)
from scipy import stats

# ── Global seed ───────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Plot aesthetics ───────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9ff',
                     'axes.edgecolor': '#cccccc', 'grid.color': '#e5e5e5'})
PALETTE = {'0': '#4C72B0', '1': '#e94560'}

print("All libraries imported successfully.")
print(f"Random seed fixed at: {SEED}")

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 3 &mdash; Load Dataset</h2>
  <p style="color:#444; line-height:1.7;">Raw dataset: <strong>100,000 customer observations × 100 features</strong>. Column descriptions loaded from <code>Description.json</code> for reference.</p>
</div>

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('Telecom_customer_churn.csv')

# ── Load column descriptions ──────────────────────────────────────────────────
with open('Description.json', 'r') as f:
    col_desc = json.load(f)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target column 'churn' present: {'churn' in df.columns}")
print()
print("First 3 rows:")
df.head(3)

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 4 &mdash; Data Understanding</h2>
  <p style="color:#444; line-height:1.7;">Before any analysis: data types, missing rates, cardinality, target distribution, and descriptive statistics. This frames every decision made downstream.</p>
</div>

In [ ]:
# ── Data types summary ────────────────────────────────────────────────────────
print("=" * 60)
print("DATA TYPES")
print("=" * 60)
dtype_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'n_unique': df.nunique()
})
print(dtype_summary.to_string())

In [ ]:
# ── Target class distribution ─────────────────────────────────────────────────
print("\nTARGET CLASS DISTRIBUTION")
print("=" * 40)
churn_counts = df['churn'].value_counts()
churn_pct = df['churn'].value_counts(normalize=True) * 100
print(f"  Not Churned (0): {churn_counts[0]:>7,}  ({churn_pct[0]:.1f}%)")
print(f"  Churned     (1): {churn_counts[1]:>7,}  ({churn_pct[1]:.1f}%)")
print(f"  Imbalance ratio: {churn_counts[0]/churn_counts[1]:.1f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count bar
axes[0].bar(['Not Churned (0)', 'Churned (1)'], churn_counts.values,
            color=['#4C72B0', '#e94560'], edgecolor='white', linewidth=1.5)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Class Distribution — Count', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Number of Customers')

# Pie
axes[1].pie(churn_counts.values, labels=['Not Churned', 'Churned'],
            colors=['#4C72B0', '#e94560'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution — Proportion', fontweight='bold', fontsize=13)

plt.suptitle('Target Variable: Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
print("NUMERICAL FEATURES — DESCRIPTIVE STATISTICS")
print("=" * 60)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"Numerical columns : {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
print()
df[num_cols].describe().T.round(3)

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 5 &mdash; Exploratory Data Analysis</h2>
  <p style="color:#444; line-height:1.7;">EDA is performed <em>before</em> cleaning — cleaning what you haven't seen is blind surgery. Covers: (5.1) Univariate, (5.2) Bivariate vs. churn, (5.3) Statistical tests, (5.4) Correlation analysis. All findings directly inform Sections 6 and 7.</p>
</div>

<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">5.1 — Univariate Analysis</h3>
  <p style="color:#555; margin:0; line-height:1.6;">Box plots for key numerical features reveal distribution shape and outlier extent. Bar charts show category frequencies for categoricals.</p>
</div>

In [ ]:
# ── 5.1.1 Missing value visualisation ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
missing_data = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_nonzero = missing_data[missing_data > 0]
if len(missing_nonzero) > 0:
    colors = ['#e94560' if x > 40 else '#f5a623' if x > 20 else '#4C72B0'
              for x in missing_nonzero.values]
    ax.bar(range(len(missing_nonzero)), missing_nonzero.values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(missing_nonzero)))
    ax.set_xticklabels(missing_nonzero.index, rotation=45, ha='right', fontsize=9)
    ax.axhline(y=40, color='#e94560', linestyle='--', alpha=0.7, label='Drop threshold (40%)')
    ax.axhline(y=20, color='#f5a623', linestyle='--', alpha=0.7, label='Unknown fill (20-40%)')
    ax.set_title('Missing Value Percentages by Column', fontweight='bold', fontsize=13)
    ax.set_ylabel('Missing (%)')
    ax.legend()
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#e94560', label='>40%: Drop column'),
                       Patch(facecolor='#f5a623', label='20-40%: Fill Unknown'),
                       Patch(facecolor='#4C72B0', label='<20%: Impute')]
    ax.legend(handles=legend_elements, loc='upper right')
else:
    ax.text(0.5, 0.5, 'No missing values detected', transform=ax.transAxes,
            ha='center', va='center', fontsize=14)
    ax.set_title('Missing Value Analysis', fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Columns with missing values: {len(missing_nonzero)}")
print(missing_nonzero[missing_nonzero > 0].to_string())

In [ ]:
# ── 5.1.2 Box plots for key numerical features ────────────────────────────────
key_num = ['rev_Mean', 'mou_Mean', 'totmrc_Mean', 'ovrmou_Mean', 'custcare_Mean',
           'drop_vce_Mean', 'months', 'hnd_price', 'eqpdays', 'avgrev']
key_num = [c for c in key_num if c in df.columns]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(key_num):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                    medianprops=dict(color='#e94560', linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor='#e94560',
                                    alpha=0.3, markersize=3))
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='both', labelsize=8)
for j in range(len(key_num), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Box Plots — Key Numerical Features (before cleaning)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.1.3 Bar charts for categorical features ────────────────────────────────
cat_plot_cols = ['new_cell', 'crclscod', 'asl_flag', 'prizm_social_one',
                 'dualband', 'refurb_new', 'hnd_webcap', 'ownrent',
                 'dwlltype', 'marital', 'creditcd']
cat_plot_cols = [c for c in cat_plot_cols if c in df.columns]

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()
for i, col in enumerate(cat_plot_cols):
    vc = df[col].value_counts().head(12)
    axes[i].bar(vc.index.astype(str), vc.values,
                color=sns.color_palette('muted', len(vc)), edgecolor='white')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)
    axes[i].tick_params(axis='y', labelsize=8)
    for spine in axes[i].spines.values():
        spine.set_edgecolor('#cccccc')
for j in range(len(cat_plot_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Categorical Feature Distributions', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">5.2 — Bivariate Analysis vs. Churn</h3>
  <p style="color:#555; margin:0; line-height:1.6;">Side-by-side box plots show separation for numericals; grouped bar charts show churn rates for categoricals. Separation = predictive signal.</p>
</div>

In [ ]:
# ── 5.2.1 Numerical features vs churn — side-by-side box plots ───────────────
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()
for i, col in enumerate(key_num):
    data0 = df[df['churn'] == 0][col].dropna()
    data1 = df[df['churn'] == 1][col].dropna()
    bp = axes[i].boxplot([data0, data1], patch_artist=True, widths=0.5,
                          boxprops=dict(alpha=0.75),
                          medianprops=dict(linewidth=2.5))
    bp['boxes'][0].set_facecolor('#4C72B0')
    bp['boxes'][1].set_facecolor('#e94560')
    axes[i].set_xticklabels(['Not Churned', 'Churned'], fontsize=9)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='y', labelsize=8)
for j in range(len(key_num), len(axes)):
    axes[j].set_visible(False)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='Not Churned'),
                   Patch(facecolor='#e94560', label='Churned')]
fig.legend(handles=legend_elements, loc='upper right', fontsize=10)
plt.suptitle('Numerical Features vs. Churn Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2.2 Churn rate by categorical feature ──────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(22, 14))
axes = axes.flatten()
for i, col in enumerate(cat_plot_cols):
    churn_rate = df.groupby(col)['churn'].mean().sort_values(ascending=False)
    churn_rate = churn_rate.head(10)
    bar_colors = ['#e94560' if x > 0.15 else '#4C72B0' for x in churn_rate.values]
    axes[i].bar(churn_rate.index.astype(str), churn_rate.values * 100,
                color=bar_colors, edgecolor='white')
    axes[i].axhline(y=df['churn'].mean() * 100, color='#333', linestyle='--',
                    alpha=0.6, linewidth=1.2, label='Overall avg')
    axes[i].set_title(f'{col} — Churn Rate', fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)', fontsize=8)
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)
    axes[i].tick_params(axis='y', labelsize=8)
    axes[i].yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
for j in range(len(cat_plot_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Churn Rate by Categorical Feature', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">5.3 — Statistical Significance Tests</h3>
  <p style="color:#555; margin:0; line-height:1.6;"><strong>Welch's t-test</strong> for numericals, <strong>Chi-Square</strong> for categoricals. Threshold: p &lt; 0.05.</p>
</div>

In [ ]:
# ── 5.3.1 Welch's t-test — numerical features vs churn ────────────────────────
ttest_results = []
for col in num_cols:
    if col in ['churn', 'Customer_ID']:
        continue
    g0 = df[df['churn'] == 0][col].dropna()
    g1 = df[df['churn'] == 1][col].dropna()
    if len(g0) < 2 or len(g1) < 2:
        continue
    stat, pval = stats.ttest_ind(g0, g1, equal_var=False)
    ttest_results.append({'feature': col, 't_statistic': round(stat, 4),
                          'p_value': pval, 'significant': pval < 0.05,
                          'mean_not_churned': round(g0.mean(), 4),
                          'mean_churned': round(g1.mean(), 4)})

ttest_df = pd.DataFrame(ttest_results).sort_values('p_value')
print("TOP 20 MOST SIGNIFICANT NUMERICAL FEATURES (Welch's t-test)")
print("=" * 70)
print(ttest_df.head(20).to_string(index=False))
print(f"\nSignificant features (p < 0.05): {ttest_df['significant'].sum()} / {len(ttest_df)}")

In [ ]:
# ── 5.3.2 Chi-Square test — categorical features vs churn ────────────────────
chi2_results = []
for col in cat_cols:
    if col == 'Customer_ID':
        continue
    contingency = pd.crosstab(df[col], df['churn'])
    if contingency.shape[0] < 2:
        continue
    chi2, pval, dof, expected = stats.chi2_contingency(contingency)
    chi2_results.append({'feature': col, 'chi2_statistic': round(chi2, 4),
                         'p_value': pval, 'dof': dof, 'significant': pval < 0.05})

chi2_df = pd.DataFrame(chi2_results).sort_values('p_value')
print("CHI-SQUARE TEST RESULTS — CATEGORICAL FEATURES vs CHURN")
print("=" * 60)
print(chi2_df.to_string(index=False))
print(f"\nSignificant features (p < 0.05): {chi2_df['significant'].sum()} / {len(chi2_df)}")

<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">5.4 — Correlation Analysis</h3>
  <p style="color:#555; margin:0; line-height:1.6;">Pearson heatmap on the top 20 most churn-correlated features. Pairs with |r| &gt; 0.90 flagged for removal in feature selection.</p>
</div>

In [ ]:
# ── 5.4 Correlation heatmap — top 20 features by churn correlation ────────────
corr_matrix = df[num_cols].corr()

# Select top 20 features most correlated with churn
top20_feats = corr_matrix['churn'].abs().drop('churn', errors='ignore').nlargest(20).index.tolist()
top20_feats = top20_feats + ['churn']

corr_sub = df[top20_feats].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.zeros_like(corr_sub, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_sub, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8},
            ax=ax, annot_kws={'size': 8})
ax.set_title('Pearson Correlation Heatmap — Top 20 Features + Churn',
             fontsize=14, fontweight='bold', pad=16)
plt.tight_layout()
plt.show()

# Highlight highly correlated pairs
high_corr_pairs = []
corr_all = df[num_cols].corr().abs()
for i in range(len(corr_all.columns)):
    for j in range(i+1, len(corr_all.columns)):
        if corr_all.iloc[i, j] > 0.90:
            high_corr_pairs.append((corr_all.columns[i], corr_all.columns[j],
                                    round(corr_all.iloc[i, j], 3)))

print(f"\nHighly correlated pairs (|r| > 0.90): {len(high_corr_pairs)}")
for a, b, r in high_corr_pairs[:15]:
    print(f"  {a:30s} <-> {b:30s}  r={r}")

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 6 &mdash; Data Cleaning</h2>
  <p style="color:#444; line-height:1.7;">Decision matrix: drop &gt;40% missing, fill Unknown for cat 20–40%, median for skewed numericals, mean for normal. Winsorize usage and revenue at 1st–99th percentile — extreme telecom users are real, not errors.</p>
</div>

In [ ]:
# ── Working copy ──────────────────────────────────────────────────────────────
df_clean = df.copy()
print(f"Starting shape: {df_clean.shape}")
print(f"Missing cells : {df_clean.isnull().sum().sum():,}")

In [ ]:
# ── Step 1: Drop Customer_ID (not a feature) ──────────────────────────────────
if 'Customer_ID' in df_clean.columns:
    df_clean.drop(columns=['Customer_ID'], inplace=True)
    print("Dropped: Customer_ID (identifier, not a predictor)")

# ── Step 2: Drop columns with >40% missing ────────────────────────────────────
missing_pct = df_clean.isnull().sum() / len(df_clean) * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
if cols_to_drop:
    df_clean.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped {len(cols_to_drop)} column(s) with >40% missing: {cols_to_drop}")
else:
    print("No columns exceed 40% missing — none dropped")

print(f"Shape after step 2: {df_clean.shape}")
print(f"Missing cells     : {df_clean.isnull().sum().sum():,}")

In [ ]:
# ── Step 3: Impute remaining missing values ───────────────────────────────────
# Re-identify column types after drops
num_cols_c = df_clean.select_dtypes(include=[np.number]).columns.drop('churn', errors='ignore').tolist()
cat_cols_c = df_clean.select_dtypes(include=['object', 'string']).columns.tolist()

filled_log = []

for col in cat_cols_c:
    null_pct = df_clean[col].isnull().mean() * 100
    if null_pct == 0:
        continue
    if null_pct > 20:
        df_clean[col] = df_clean[col].fillna('Unknown')
        filled_log.append((col, f'{null_pct:.1f}%', 'Unknown label'))
    else:
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        filled_log.append((col, f'{null_pct:.1f}%', f'Mode ({mode_val})'))

for col in num_cols_c:
    null_pct = df_clean[col].isnull().mean() * 100
    if null_pct == 0:
        continue
    skewness = df_clean[col].skew()
    if abs(skewness) > 1:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        filled_log.append((col, f'{null_pct:.1f}%', f'Median ({median_val:.2f}) — skew={skewness:.2f}'))
    else:
        mean_val = df_clean[col].mean()
        df_clean[col] = df_clean[col].fillna(mean_val)
        filled_log.append((col, f'{null_pct:.1f}%', f'Mean ({mean_val:.2f}) — skew={skewness:.2f}'))

print("IMPUTATION LOG")
print("-" * 70)
for col, pct, method in filled_log:
    print(f"  {col:35s}  {pct:>6s} missing  ->  {method}")
print(f"\nShape after imputation: {df_clean.shape}")
print(f"Remaining missing cells: {df_clean.isnull().sum().sum()}")

In [ ]:
# ── Step 4: Winsorize usage and revenue columns ───────────────────────────────
winsor_keywords = ['_Mean', 'rev', 'mou', 'totmou', 'totrev', 'adjrev', 'adjmou',
                   'avgrev', 'avgmou', 'avg3mou', 'avg6mou', 'hnd_price']
winsor_cols = [c for c in num_cols_c if any(kw in c for kw in winsor_keywords)]

from scipy.stats.mstats import winsorize as scipy_winsorize

for col in winsor_cols:
    df_clean[col] = scipy_winsorize(df_clean[col].values, limits=[0.01, 0.01])

print(f"Winsorized {len(winsor_cols)} usage/revenue columns at 1st-99th percentile")
print(f"Final clean shape: {df_clean.shape}")
print(f"Missing cells    : {df_clean.isnull().sum().sum()}")

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 7 &mdash; Feature Engineering</h2>
  <p style="color:#444; line-height:1.7;">8 derived features encode domain knowledge the raw columns miss: call completion rate, overage ratio, revenue per minute, high care usage flag, drop-block rate, revenue trend flag, tenure group, equipment age group. Encoding: ordinal → LabelEncoder, nominal → one-hot, high-cardinality → frequency encoding.</p>
</div>

In [ ]:
# ── 7.1 Derived numerical features ───────────────────────────────────────────
df_fe = df_clean.copy()

# Call completion rate: failed calls = frustration = churn risk
if 'comp_vce_Mean' in df_fe and 'plcd_vce_Mean' in df_fe:
    df_fe['call_completion_rate'] = df_fe['comp_vce_Mean'] / (df_fe['plcd_vce_Mean'] + 1)

# Overage ratio: being charged extra = unhappy customer
if 'ovrmou_Mean' in df_fe and 'mou_Mean' in df_fe:
    df_fe['overage_ratio'] = df_fe['ovrmou_Mean'] / (df_fe['mou_Mean'] + 1)

# Revenue per minute: value efficiency signal
if 'rev_Mean' in df_fe and 'mou_Mean' in df_fe:
    df_fe['rev_per_minute'] = df_fe['rev_Mean'] / (df_fe['mou_Mean'] + 1)

# High customer care usage: frequent complaints = dissatisfied
if 'custcare_Mean' in df_fe:
    q75 = df_fe['custcare_Mean'].quantile(0.75)
    df_fe['high_custcare'] = (df_fe['custcare_Mean'] > q75).astype(int)

# Drop-block rate: network quality signal
if 'drop_blk_Mean' in df_fe and 'attempt_Mean' in df_fe:
    df_fe['drop_block_rate'] = df_fe['drop_blk_Mean'] / (df_fe['attempt_Mean'] + 1)

# Revenue trending down: decline before churn
if 'change_rev' in df_fe:
    df_fe['rev_trending_down'] = (df_fe['change_rev'] < 0).astype(int)

# Tenure group: non-linear tenure effect
if 'months' in df_fe:
    df_fe['tenure_group'] = pd.cut(df_fe['months'],
                                    bins=[0, 6, 12, 24, 48, np.inf],
                                    labels=['0-6m', '6-12m', '12-24m', '24-48m', '48m+'])

# Equipment age group: device age signal
if 'eqpdays' in df_fe:
    df_fe['eqp_age_group'] = pd.cut(df_fe['eqpdays'],
                                     bins=[0, 180, 365, 730, np.inf],
                                     labels=['<6m', '6m-1y', '1-2y', '2y+'])

new_features = ['call_completion_rate', 'overage_ratio', 'rev_per_minute',
                'high_custcare', 'drop_block_rate', 'rev_trending_down',
                'tenure_group', 'eqp_age_group']
existing_new = [f for f in new_features if f in df_fe.columns]
print(f"Created {len(existing_new)} engineered features: {existing_new}")
print(f"Shape: {df_fe.shape}")

In [ ]:
# ── 7.2 Encode categorical features ──────────────────────────────────────────
le = LabelEncoder()

# Ordinal — tenure_group (natural order already embedded in cut labels)
if 'tenure_group' in df_fe.columns:
    df_fe['tenure_group'] = le.fit_transform(df_fe['tenure_group'].astype(str))
    print("LabelEncoded: tenure_group (ordinal)")

# Ordinal — eqp_age_group
if 'eqp_age_group' in df_fe.columns:
    df_fe['eqp_age_group'] = le.fit_transform(df_fe['eqp_age_group'].astype(str))
    print("LabelEncoded: eqp_age_group (ordinal)")

# Ordinal — crclscod (credit class: roughly ordinal A > B > C...)
if 'crclscod' in df_fe.columns:
    df_fe['crclscod'] = le.fit_transform(df_fe['crclscod'].astype(str))
    print("LabelEncoded: crclscod (credit class — treated as ordinal)")

# High-cardinality nominal → frequency encoding
high_card_cols = [c for c in df_fe.select_dtypes(include=['object','string']).columns
                  if df_fe[c].nunique() > 15]
for col in high_card_cols:
    freq_map = df_fe[col].value_counts(normalize=True).to_dict()
    df_fe[col] = df_fe[col].map(freq_map)
    print(f"FrequencyEncoded: {col} ({df_fe[col].nunique()} unique before encoding)")

# Nominal → get_dummies (drop_first=True to avoid multicollinearity)
remaining_cats = df_fe.select_dtypes(include=['object','string']).columns.tolist()
nominal_to_encode = [c for c in remaining_cats if c != 'churn']
if nominal_to_encode:
    df_fe = pd.get_dummies(df_fe, columns=nominal_to_encode, drop_first=True, dtype=int)
    print(f"OneHotEncoded: {nominal_to_encode}")

print(f"\nFinal shape after encoding: {df_fe.shape}")

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 8 &mdash; Feature Selection</h2>
  <p style="color:#444; line-height:1.7;">Four-stage funnel: (1) VarianceThreshold(0.01) removes near-zero variance, (2) Correlation filter |r|&gt;0.90 removes redundant pairs, (3) Mutual Information top 40 — model-agnostic, captures non-linear relationships, (4) Random Forest importance top 25 — final selection.</p>
</div>

In [ ]:
# ── Prepare X and y ───────────────────────────────────────────────────────────
X = df_fe.drop(columns=['churn'])
y = df_fe['churn'].astype(int)

print(f"Pre-selection: X shape = {X.shape}")

# ── Stage 1: Variance threshold ───────────────────────────────────────────────
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)
selected_after_vt = X.columns[vt.get_support()].tolist()
print(f"After VarianceThreshold : {len(selected_after_vt)} features (removed {X.shape[1] - len(selected_after_vt)})")

In [ ]:
# ── Stage 2: Correlation filter ──────────────────────────────────────────────
X_vt_df = pd.DataFrame(X_vt, columns=selected_after_vt)
corr_mat = X_vt_df.corr().abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
cols_to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.90)]
X_corr_df = X_vt_df.drop(columns=cols_to_drop_corr)
print(f"After Correlation filter: {X_corr_df.shape[1]} features (removed {len(cols_to_drop_corr)})")
print(f"  Removed: {cols_to_drop_corr[:10]}{'...' if len(cols_to_drop_corr)>10 else ''}")

In [ ]:
# ── Stage 3: Mutual information — top 40 ─────────────────────────────────────
mi_scores = mutual_info_classif(X_corr_df, y, random_state=SEED)
mi_series = pd.Series(mi_scores, index=X_corr_df.columns).sort_values(ascending=False)
top40 = mi_series.head(40).index.tolist()
X_mi_df = X_corr_df[top40]
print(f"After Mutual Info (top 40): {X_mi_df.shape[1]} features")

# ── MI score bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#e94560' if i < 10 else '#4C72B0' for i in range(40)]
ax.barh(range(40), mi_series.head(40).values[::-1], color=colors[::-1], edgecolor='white')
ax.set_yticks(range(40))
ax.set_yticklabels(mi_series.head(40).index[::-1], fontsize=8)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Mutual Information Scores — Top 40 Features', fontweight='bold', fontsize=13)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#e94560', label='Top 10'),
                   Patch(facecolor='#4C72B0', label='Positions 11-40')], fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Stage 4: Random Forest importance — top 25 ───────────────────────────────
rf_sel = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_sel.fit(X_mi_df, y)
rf_imp = pd.Series(rf_sel.feature_importances_, index=X_mi_df.columns).sort_values(ascending=False)
top25 = rf_imp.head(25).index.tolist()
X_final = X_mi_df[top25]
print(f"After RF Importance (top 25): {X_final.shape[1]} features")
print(f"Final selected features: {top25}")

# ── RF importance bar chart ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
palette = sns.color_palette('Blues_r', n_colors=25)
ax.barh(range(25), rf_imp.head(25).values[::-1], color=palette[::-1], edgecolor='white')
ax.set_yticks(range(25))
ax.set_yticklabels(rf_imp.head(25).index[::-1], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importance — Final Top 25', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 9 &mdash; Train / Test Split</h2>
  <p style="color:#444; line-height:1.7;">80/20 stratified split preserves the churn ratio in both subsets. SMOTE and scaling applied <em>after</em> this split, on training data only — no leakage.</p>
</div>

In [ ]:
# ── Stratified split 80 / 20 ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.20, stratify=y, random_state=SEED
)

print("SPLIT RESULTS")
print("=" * 40)
print(f"  Train: {X_train.shape[0]:,} rows")
print(f"  Test : {X_test.shape[0]:,} rows")
print()
print("CLASS DISTRIBUTION VERIFICATION")
print(f"  Train churn rate: {y_train.mean():.4f}  ({y_train.sum():,} churners)")
print(f"  Test  churn rate: {y_test.mean():.4f}  ({y_test.sum():,} churners)")
print()
diff = abs(y_train.mean() - y_test.mean())
print(f"  Rate difference : {diff:.5f}  {'PASS — stratification confirmed' if diff < 0.005 else 'WARN — check stratification'}") 

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 10 &mdash; Handle Class Imbalance (SMOTE)</h2>
  <p style="color:#444; line-height:1.7;">If minority class &lt;30% of training data, SMOTE synthesises new minority samples from k-nearest neighbours — training set only. Otherwise class_weight='balanced' inside each model.</p>
</div>

In [ ]:
# ── SMOTE decision ────────────────────────────────────────────────────────────
churn_rate_train = y_train.mean()
print(f"Training churn rate: {churn_rate_train:.4f}")

USE_SMOTE = churn_rate_train < 0.30

if USE_SMOTE and SMOTE_AVAILABLE:
    print("Applying SMOTE (churn rate < 0.30)...")
    sm = SMOTE(random_state=SEED, k_neighbors=5)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print(f"  Before: {y_train.value_counts().to_dict()}")
    print(f"  After : {pd.Series(y_train_res).value_counts().to_dict()}")
    print(f"  New training size: {len(X_train_res):,}")
elif USE_SMOTE and not SMOTE_AVAILABLE:
    print("SMOTE requested but imbalanced-learn not available.")
    print("Using class_weight='balanced' in all models instead.")
    X_train_res, y_train_res = X_train.copy(), y_train.copy()
    USE_SMOTE = False
else:
    print("Churn rate >= 0.30 — using class_weight='balanced' in models instead.")
    X_train_res, y_train_res = X_train.copy(), y_train.copy()

print(f"\nFinal training class distribution:")
print(pd.Series(y_train_res).value_counts().rename({0: 'Not Churned', 1: 'Churned'}).to_string())

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 11 &mdash; Scaling</h2>
  <p style="color:#444; line-height:1.7;">Two data versions: <code>X_train_scaled / X_test_scaled</code> for LR and KNN; <code>X_train_tree / X_test_tree</code> (unscaled) for tree models. StandardScaler fitted on <strong>training data only</strong>.</p>
</div>

In [ ]:
# ── StandardScaler fit on training only ───────────────────────────────────────
scaler = StandardScaler()

# Scaled versions — for linear models and KNN
X_train_scaled = scaler.fit_transform(X_train_res)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)            # transform only on test

# Tree versions — unscaled (tree-based models don't need scaling)
X_train_tree = X_train_res.values if hasattr(X_train_res, 'values') else X_train_res
X_test_tree  = X_test.values  if hasattr(X_test, 'values')  else X_test

y_train_final = y_train_res.values if hasattr(y_train_res, 'values') else np.array(y_train_res)
y_test_arr    = y_test.values  if hasattr(y_test, 'values')  else np.array(y_test)

print("Scaling complete.")
print(f"  X_train_scaled shape : {X_train_scaled.shape}")
print(f"  X_test_scaled  shape : {X_test_scaled.shape}")
print(f"  X_train_tree   shape : {X_train_tree.shape}")
print(f"  X_test_tree    shape : {X_test_tree.shape}")
print(f"  Scaler mean[0:3] fitted on train: {scaler.mean_[:3].round(4)}")

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 12 &mdash; Baseline Model — Logistic Regression</h2>
  <p style="color:#444; line-height:1.7;">Interpretable baseline. Coefficients encode feature effects as log-odds. Benchmarks all subsequent models.</p>
</div>

In [ ]:
# ── Helper: evaluate model ────────────────────────────────────────────────────
from sklearn.decomposition import PCA

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train, predict, compute all metrics, plot confusion matrix + class distribution."""
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    auc  = roc_auc_score(y_te, y_proba) if y_proba is not None else 0.0

    cm   = confusion_matrix(y_te, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Train F1 for overfitting check
    y_tr_pred = model.predict(X_tr)
    train_f1  = f1_score(y_tr, y_tr_pred, zero_division=0)
    gap = train_f1 - f1

    print(f"\n{'='*60}")
    print(f"MODEL: {name}")
    print(f"{'='*60}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall      : {rec:.4f}   <- PRIMARY METRIC")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1 Score    : {f1:.4f}")
    print(f"  AUC-ROC     : {auc:.4f}")
    print(f"  Train F1    : {train_f1:.4f} | Test F1: {f1:.4f} | Gap: {gap:.4f}")
    if gap > 0.10:
        print(f"  WARNING: Overfitting gap > 0.10 — consider regularisation or pruning")
    else:
        print(f"  Overfitting check: PASS (gap <= 0.10)")

    # ── Figure: Confusion Matrix + PCA Class Distribution (side by side) ──────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('white')

    # Left: Confusion Matrix
    disp = ConfusionMatrixDisplay(cm, display_labels=['Not Churned', 'Churned'])
    disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'{name}\nConfusion Matrix', fontweight='bold', fontsize=12)

    # Right: PCA 2D class distribution (predicted classes, style of reference image)
    X_te_arr = X_te if isinstance(X_te, np.ndarray) else np.array(X_te)

    # Use PCA to reduce to 2 dimensions for visualisation
    pca_vis = PCA(n_components=2, random_state=42)
    X_2d = pca_vis.fit_transform(X_te_arr)

    var_exp = pca_vis.explained_variance_ratio_ * 100

    # Plot predicted classes
    colors_pred = {0: '#4C72B0', 1: '#e94560'}
    labels_pred = {0: 'Predicted: Not Churned', 1: 'Predicted: Churned'}
    markers_pred = {0: 'o', 1: 's'}

    for cls in [0, 1]:
        mask = y_pred == cls
        axes[1].scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            c=colors_pred[cls], label=labels_pred[cls],
            alpha=0.45, s=18, marker=markers_pred[cls],
            edgecolors='none'
        )

    # Draw a smooth decision boundary approximation using a grid
    x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
    y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120),
                         np.linspace(y_min, y_max, 120))

    # Project grid back to original space and predict
    grid_orig = pca_vis.inverse_transform(np.c_[xx.ravel(), yy.ravel()])
    Z = model.predict(grid_orig)
    Z = Z.reshape(xx.shape)

    axes[1].contourf(xx, yy, Z, alpha=0.08,
                     levels=[-0.5, 0.5, 1.5],
                     colors=['#4C72B0', '#e94560'])
    axes[1].contour(xx, yy, Z, levels=[0.5], colors=['#111111'],
                    linewidths=1.8, linestyles='-')

    axes[1].set_xlabel(f'PCA Component 1 ({var_exp[0]:.1f}% variance)', fontsize=10)
    axes[1].set_ylabel(f'PCA Component 2 ({var_exp[1]:.1f}% variance)', fontsize=10)
    axes[1].set_title(f'{name}\nPredicted Class Distribution (PCA 2D)', fontweight='bold', fontsize=12)
    axes[1].legend(loc='upper right', fontsize=9, framealpha=0.85,
                   markerscale=1.5)
    axes[1].set_facecolor('#f8f9ff')

    # Annotation: class counts
    n0 = (y_pred == 0).sum()
    n1 = (y_pred == 1).sum()
    axes[1].annotate(
        f'Not Churned: {n0:,}  |  Churned: {n1:,}',
        xy=(0.5, -0.13), xycoords='axes fraction',
        ha='center', fontsize=9, color='#555555',
        style='italic'
    )

    plt.suptitle(f'{name} — Model Output Visualisation', fontsize=13,
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    return {
        'model': model, 'name': name,
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'specificity': spec, 'f1': f1, 'auc': auc,
        'train_f1': train_f1, 'gap': gap,
        'y_pred': y_pred, 'y_proba': y_proba
    }

results = {}  # stores all model results

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────────────────
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=SEED,
    C=1.0
)
results['Logistic Regression'] = evaluate_model(
    'Logistic Regression', lr,
    X_train_scaled, X_test_scaled,
    y_train_final, y_test_arr
)

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 13 &mdash; Train Multiple Models</h2>
  <p style="color:#444; line-height:1.7;">Five model families in order of complexity, each with the correct data version. Decision Tree for interpretable rules, KNN for local pattern detection, Random Forest via bagging, XGBoost via gradient boosting.</p>
</div>

In [ ]:
# ── Model 2: Decision Tree ────────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=SEED)
results['Decision Tree'] = evaluate_model(
    'Decision Tree', dt,
    X_train_tree, X_test_tree,
    y_train_final, y_test_arr
)

In [ ]:
# ── Model 3: K-Nearest Neighbours ────────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=11, n_jobs=-1)
results['KNN'] = evaluate_model(
    'KNN', knn,
    X_train_scaled, X_test_scaled,
    y_train_final, y_test_arr
)

In [ ]:
# ── Model 4: Random Forest ────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)
results['Random Forest'] = evaluate_model(
    'Random Forest', rf,
    X_train_tree, X_test_tree,
    y_train_final, y_test_arr
)

In [ ]:
# ── Model 5: XGBoost ─────────────────────────────────────────────────────────
if XGB_AVAILABLE:
    scale_pos = int((y_train_final == 0).sum() / (y_train_final == 1).sum())
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=scale_pos,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=SEED,
        n_jobs=-1
    )
    results['XGBoost'] = evaluate_model(
        'XGBoost', xgb,
        X_train_tree, X_test_tree,
        y_train_final, y_test_arr
    )
else:
    gb = GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05, random_state=SEED
    )
    results['GradientBoosting'] = evaluate_model(
        'GradientBoosting', gb,
        X_train_tree, X_test_tree,
        y_train_final, y_test_arr
    )

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 14 &mdash; Hyperparameter Tuning</h2>
  <p style="color:#444; line-height:1.7;">RandomizedSearchCV (25 iterations × 5-fold StratifiedKFold) on RF and XGBoost. Optimisation target: F1 score. Best parameters are recorded and the tuned models added to the shared <code>results</code> dict.</p>
</div>

In [ ]:
# ── Select best model from Section 13 ────────────────────────────────────────
best_name = max(results, key=lambda k: results[k]['f1'])
print(f"Best model by F1: {best_name} (F1 = {results[best_name]['f1']:.4f})")
print("Proceeding with hyperparameter tuning on Random Forest and XGBoost/GB...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
# ── Tune Random Forest ────────────────────────────────────────────────────────
rf_param_dist = {
    'n_estimators'      : [100, 200, 300, 400],
    'max_depth'         : [6, 8, 10, 12, 15, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
    'max_features'      : ['sqrt', 'log2', 0.5],
    'class_weight'      : ['balanced', 'balanced_subsample']
}

rf_tuned_base = RandomForestClassifier(random_state=SEED, n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_tuned_base, rf_param_dist, n_iter=25,
    scoring='f1', cv=cv, random_state=SEED, n_jobs=-1, verbose=0
)
rf_search.fit(X_train_tree, y_train_final)

print(f"RF Best params : {rf_search.best_params_}")
print(f"RF Best CV F1  : {rf_search.best_score_:.4f}")

results['RF Tuned'] = evaluate_model(
    'RF Tuned', rf_search.best_estimator_,
    X_train_tree, X_test_tree,
    y_train_final, y_test_arr
)

In [ ]:
# ── Tune XGBoost / GradientBoosting ──────────────────────────────────────────
if XGB_AVAILABLE:
    xgb_param_dist = {
        'n_estimators'    : [100, 200, 300],
        'max_depth'       : [3, 4, 5, 6, 8],
        'learning_rate'   : [0.01, 0.03, 0.05, 0.1, 0.15],
        'subsample'       : [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'gamma'           : [0, 0.1, 0.2, 0.5],
        'min_child_weight': [1, 3, 5],
        'reg_alpha'       : [0, 0.1, 0.5],
        'reg_lambda'      : [1, 1.5, 2]
    }
    xgb_base = XGBClassifier(scale_pos_weight=scale_pos, use_label_encoder=False,
                              eval_metric='logloss', random_state=SEED, n_jobs=-1)
    xgb_search = RandomizedSearchCV(
        xgb_base, xgb_param_dist, n_iter=25,
        scoring='f1', cv=cv, random_state=SEED, n_jobs=-1, verbose=0
    )
    xgb_search.fit(X_train_tree, y_train_final)
    print(f"XGB Best params: {xgb_search.best_params_}")
    print(f"XGB Best CV F1 : {xgb_search.best_score_:.4f}")
    results['XGBoost Tuned'] = evaluate_model(
        'XGBoost Tuned', xgb_search.best_estimator_,
        X_train_tree, X_test_tree,
        y_train_final, y_test_arr
    )
else:
    gb_param_dist = {
        'n_estimators' : [100, 200, 300],
        'max_depth'    : [3, 4, 5, 6],
        'learning_rate': [0.01, 0.05, 0.1, 0.15],
        'subsample'    : [0.7, 0.8, 0.9, 1.0],
        'min_samples_split': [2, 5, 10]
    }
    gb_base = GradientBoostingClassifier(random_state=SEED)
    gb_search = RandomizedSearchCV(
        gb_base, gb_param_dist, n_iter=20,
        scoring='f1', cv=cv, random_state=SEED, n_jobs=-1, verbose=0
    )
    gb_search.fit(X_train_tree, y_train_final)
    results['GB Tuned'] = evaluate_model(
        'GB Tuned', gb_search.best_estimator_,
        X_train_tree, X_test_tree,
        y_train_final, y_test_arr
    )

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 15 &mdash; Advanced Ensemble Methods</h2>
  <p style="color:#444; line-height:1.7;">Three techniques push beyond any single model: <strong>LightGBM</strong> (leaf-wise gradient boosting, 10–20× faster than XGBoost on wide data), <strong>Soft Voting</strong> (RF + XGBoost + LightGBM probability averaging), <strong>Stacking</strong> (four base models → LR meta-learner trained on out-of-fold predictions). All results write into the shared <code>results</code> dict. Targets: F1&gt;0.68 | AUC&gt;0.72 | Recall&gt;0.78.</p>
</div>

In [ ]:
# ── New imports for ensemble methods ─────────────────────────────────────────
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.decomposition import PCA
from matplotlib.patches import FancyBboxPatch

# ── evaluate_bonus() — uses shared variables from Sections 9–11 ───────────────
def evaluate_bonus(name, model, X_te, y_te, X_tr=None, y_tr=None):
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    spec = recall_score(y_te, y_pred, pos_label=0, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    auc  = roc_auc_score(y_te, y_proba) if y_proba is not None else 0.0
    cm   = confusion_matrix(y_te, y_pred)

    gap = None
    if X_tr is not None and y_tr is not None:
        y_pred_tr = model.predict(X_tr)
        gap = f1_score(y_tr, y_pred_tr, zero_division=0) - f1

    print(f"\n{'='*58}")
    print(f"  MODEL: {name}")
    print(f"{'='*58}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall      : {rec:.4f}   <- PRIMARY METRIC")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1 Score    : {f1:.4f}")
    print(f"  AUC-ROC     : {auc:.4f}")
    if gap is not None:
        flag = "WARNING: gap > 0.10" if gap > 0.10 else "PASS"
        print(f"  Train-Test Gap: {gap:.4f}  {flag}")

    targets = {'LightGBM (Tuned)': (0.65, 0.70, 0.70),
               'Voting Ensemble':  (0.66, 0.71, 0.71),
               'Stacking':         (0.67, 0.72, 0.72),
               'Best + Threshold': (0.68, 0.72, 0.78)}
    if name in targets:
        tf1, tauc, trec = targets[name]
        print(f"\n  Target check — F1>{tf1}: {'PASS' if f1>tf1 else 'MISS'}  "
              f"AUC>{tauc}: {'PASS' if auc>tauc else 'MISS'}  "
              f"Recall>{trec}: {'PASS' if rec>trec else 'MISS'}")

    # ── Plot: CM + PCA scatter + interpretation panel ─────────────────────────
    fig = plt.figure(figsize=(18, 11))
    fig.patch.set_facecolor('white')
    ax_cm  = fig.add_subplot(2, 2, 1)
    ax_pca = fig.add_subplot(2, 2, 2)

    ConfusionMatrixDisplay(cm, display_labels=['Not Churned', 'Churned']).plot(
        ax=ax_cm, cmap='Blues', colorbar=False)
    ax_cm.set_title(f'{name}\nConfusion Matrix', fontweight='bold', fontsize=12)

    X_te_arr = X_te if isinstance(X_te, np.ndarray) else np.array(X_te)
    pca_vis  = PCA(n_components=2, random_state=42)
    X_2d     = pca_vis.fit_transform(X_te_arr)
    var_exp  = pca_vis.explained_variance_ratio_ * 100
    for cls, color, marker, lbl in [(0,'#4C72B0','o','Not Churned'),(1,'#e94560','s','Churned')]:
        mask = y_pred == cls
        ax_pca.scatter(X_2d[mask,0], X_2d[mask,1], c=color, alpha=0.4,
                       s=18, marker=marker, edgecolors='none', label=lbl)
    x_min,x_max = X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5
    y_min,y_max = X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5
    xx,yy = np.meshgrid(np.linspace(x_min,x_max,100), np.linspace(y_min,y_max,100))
    Z = model.predict(pca_vis.inverse_transform(np.c_[xx.ravel(),yy.ravel()])).reshape(xx.shape)
    ax_pca.contourf(xx,yy,Z,alpha=0.08,levels=[-0.5,0.5,1.5],colors=['#4C72B0','#e94560'])
    ax_pca.contour(xx,yy,Z,levels=[0.5],colors=['#111'],linewidths=1.8)
    ax_pca.set_xlabel(f'PCA 1 ({var_exp[0]:.1f}% var)',fontsize=10)
    ax_pca.set_ylabel(f'PCA 2 ({var_exp[1]:.1f}% var)',fontsize=10)
    ax_pca.set_title(f'{name}\nPCA 2D Decision Space',fontweight='bold',fontsize=12)
    ax_pca.legend(fontsize=9)
    ax_pca.set_facecolor('#f8f9ff')
    ax_pca.annotate(f'Not Churned:{(y_pred==0).sum():,}  |  Churned:{(y_pred==1).sum():,}',
                    xy=(0.5,-0.13),xycoords='axes fraction',ha='center',fontsize=9,color='#555',style='italic')

    ax_m = fig.add_subplot(2,1,2)
    ax_m.set_facecolor('#f8f9ff'); ax_m.set_xlim(0,1); ax_m.set_ylim(0,1); ax_m.axis('off')
    ax_m.add_patch(FancyBboxPatch((0.01,0.02),0.98,0.96,boxstyle="round,pad=0.01",
                                   facecolor='#f8f9ff',edgecolor='#dde3f0',linewidth=1.5))
    ax_m.text(0.5,0.91,f'{name} — Metrics Interpretation',ha='center',va='center',
              fontsize=13,fontweight='bold',color='#0f3460',transform=ax_m.transAxes)
    ax_m.axhline(y=0.85,xmin=0.02,xmax=0.98,color='#dde3f0',linewidth=1.2)

    def badge(v,m):
        g={'Recall':0.75,'F1 Score':0.75,'AUC-ROC':0.70,'Accuracy':0.70}.get(m,0.65)
        fair={'Recall':0.60,'F1 Score':0.60,'AUC-ROC':0.60,'Accuracy':0.55}.get(m,0.50)
        return '#2ca02c' if v>=g else '#f5a623' if v>=fair else '#e94560'

    for i,(mname,mval,mdesc) in enumerate([
        ('Accuracy',acc,'Correct/total. Misleading\non imbalanced data.'),
        ('Precision',prec,'Flagged churners\nthat were real.'),
        ('Recall',rec,'Churners caught.\nPRIMARY — miss=$200.'),
        ('Specificity',spec,'Loyal customers\ncorrectly left alone.'),
        ('F1 Score',f1,'Harmonic mean\nPrecision & Recall.'),
        ('AUC-ROC',auc,'Ranking ability.\n0.5=random 1.0=perfect.'),
    ]):
        xc=(i+0.5)/6
        ax_m.text(xc,0.81,mname,ha='center',va='center',fontsize=10,fontweight='bold',
                  color='#e94560' if mname=='Recall' else '#0f3460',transform=ax_m.transAxes)
        bc=badge(mval,mname)
        ax_m.add_patch(FancyBboxPatch((xc-0.065,0.60),0.13,0.10,boxstyle="round,pad=0.01",
                                       facecolor=bc,edgecolor='white',linewidth=1.5,zorder=2,transform=ax_m.transAxes))
        ax_m.text(xc,0.65,f'{mval:.4f}',ha='center',va='center',fontsize=13,fontweight='bold',
                  color='white',transform=ax_m.transAxes,zorder=3)
        ax_m.add_patch(FancyBboxPatch((xc-0.07,0.50),0.14,0.04,boxstyle="round,pad=0.005",
                                       facecolor='#e0e0e0',edgecolor='none',transform=ax_m.transAxes))
        ax_m.add_patch(FancyBboxPatch((xc-0.07,0.50),0.14*mval,0.04,boxstyle="round,pad=0.005",
                                       facecolor=bc,edgecolor='none',alpha=0.85,transform=ax_m.transAxes))
        for li,lt in enumerate(mdesc.split('\n')):
            ax_m.text(xc,0.36-li*0.12,lt,ha='center',va='center',fontsize=7.2,
                      color='#444',style='italic',transform=ax_m.transAxes)

    gap_val=gap if gap else 0.0
    gc='#e94560' if gap_val>0.10 else '#2ca02c'
    ax_m.text(0.5,0.07,f'Train-Test Gap: {gap_val:.4f}  — {"WARNING>0.10" if gap_val>0.10 else "PASS<=0.10"}',
              ha='center',va='center',fontsize=9,fontweight='bold',color=gc,transform=ax_m.transAxes)
    plt.suptitle(f'{name} — Output Visualisation',fontsize=13,fontweight='bold',y=1.01)
    plt.tight_layout(); plt.show()

    return {'model':model,'name':name,'accuracy':acc,'precision':prec,'recall':rec,
            'specificity':spec,'f1':f1,'auc':auc,
            'train_f1':(f1+gap) if gap else f1,'gap':gap if gap else 0.0,
            'y_pred':y_pred,'y_proba':y_proba}

print("evaluate_bonus() ready — all Section 9–11 variables available.")


<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">15.1 — LightGBM (Tuned)</h3>
  <p style="color:#555; margin:0; line-height:1.6;">Leaf-wise tree growth + histogram binning = 10–20× faster than XGBoost on 100 features, often finding better splits. Key parameter: <code>num_leaves</code> controls complexity independently of depth. 50 iterations × 5-fold CV.</p>
</div>

In [ ]:
# ── Install LightGBM if needed ────────────────────────────────────────────────
try:
    from lightgbm import LGBMClassifier
    print("LightGBM available.")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'])
    from lightgbm import LGBMClassifier
    print("LightGBM installed and imported.")

In [ ]:
# ── B1: LightGBM with RandomizedSearchCV ──────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgbm_base = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    class_weight='balanced'
)

lgbm_param_dist = {
    'n_estimators'     : [300, 500, 700, 1000],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.1],
    'num_leaves'       : [31, 63, 127],
    'max_depth'        : [-1, 6, 8, 10],
    'min_child_samples': [20, 30, 50],
    'subsample'        : [0.7, 0.8, 1.0],
    'colsample_bytree' : [0.7, 0.8, 1.0],
    'reg_alpha'        : [0, 0.1, 0.5, 1.0],
    'reg_lambda'       : [0.5, 1.0, 2.0],
}

print("Running LightGBM RandomizedSearchCV (50 iterations x 5 folds)...")
lgbm_search = RandomizedSearchCV(
    lgbm_base,
    param_distributions=lgbm_param_dist,
    n_iter=50,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

lgbm_search.fit(X_train_tree, y_train_final)
lgbm_best = lgbm_search.best_estimator_

print(f"\nBest CV F1  : {lgbm_search.best_score_:.4f}")
print(f"Best params : {lgbm_search.best_params_}")

In [ ]:
# ── B1: Evaluate LightGBM ─────────────────────────────────────────────────────
lgbm_results = evaluate_bonus(
    'LightGBM (Tuned)', lgbm_best,
    X_test_tree, y_test_arr,
    X_train_tree, y_train_final
)
results['LightGBM (Tuned)'] = lgbm_results

<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">15.2 — Soft Voting Ensemble (RF + XGBoost + LightGBM)</h3>
  <p style="color:#555; margin:0; line-height:1.6;">Averages predicted <em>probabilities</em> — preserves model confidence. RF (bagging), XGBoost (level-wise), LightGBM (leaf-wise) make structurally different errors. <code>weights=[1,2,2]</code> reflects booster superiority over RF.</p>
</div>

In [ ]:
# ── B2: Soft Voting Ensemble ──────────────────────────────────────────────────
# Combine the 3 strongest tree-based models: RF Tuned, XGBoost, LightGBM
# All are tree-based — use X_train_tree throughout, no scaling needed

voting_clf = VotingClassifier(
    estimators=[
        ('rf',   results['RF Tuned']['model']),
        ('xgb',  results['XGBoost']['model']),
        ('lgbm', lgbm_best),
    ],
    voting='soft',
    weights=[1, 2, 2],   # stronger models get more weight
    n_jobs=-1
)

print("Training Soft Voting Ensemble (RF + XGBoost + LightGBM)...")
voting_clf.fit(X_train_tree, y_train_final)
print("Training complete.")

In [ ]:
# ── B2: Evaluate Voting Ensemble ──────────────────────────────────────────────
voting_results = evaluate_bonus(
    'Voting Ensemble', voting_clf,
    X_test_tree, y_test_arr,
    X_train_tree, y_train_final
)
results['Voting Ensemble'] = voting_results

# Side-by-side comparison vs best single model
print("\n" + "─"*55)
print("  COMPARISON: Voting Ensemble vs Best Single (XGBoost)")
print("─"*55)
for metric in ['accuracy','precision','recall','specificity','f1','auc']:
    base_val  = results['XGBoost'][metric]
    bonus_val = voting_results[metric]
    delta     = bonus_val - base_val
    sign      = '+' if delta >= 0 else ''
    flag      = ' ↑' if delta > 0.005 else (' ↓' if delta < -0.005 else '')
    print(f"  {metric.capitalize():12s}: XGBoost={base_val:.4f}  Voting={bonus_val:.4f}  ({sign}{delta:.4f}){flag}")

<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">15.3 — Stacking (DT + RF + XGBoost + LightGBM → LR meta)</h3>
  <p style="color:#555; margin:0; line-height:1.6;">Meta-learner trained on out-of-fold predictions (StratifiedKFold, 5 splits) — learns the optimal combination from data, not fixed weights. LR meta-learner: interpretable coefficients, auditable. Result: meta-learner tells us which model to trust per customer segment.</p>
</div>

In [ ]:
# ── B3: Stacking Classifier ───────────────────────────────────────────────────
stacking_clf = StackingClassifier(
    estimators=[
        ('dt',   results['Decision Tree']['model']),
        ('rf',   results['RF Tuned']['model']),
        ('xgb',  results['XGBoost']['model']),
        ('lgbm', lgbm_best),
    ],
    final_estimator=LogisticRegression(
        C=1.0, max_iter=1000, random_state=42, class_weight='balanced'
    ),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method='predict_proba',
    n_jobs=-1
)

print("Training Stacking Classifier (DT + RF + XGBoost + LightGBM → LR meta)...")
print("This uses 5-fold CV to generate out-of-fold meta-features — takes a few minutes...")
stacking_clf.fit(X_train_tree, y_train_final)
print("Training complete.")

In [ ]:
# ── B3: Evaluate Stacking ─────────────────────────────────────────────────────
stacking_results = evaluate_bonus(
    'Stacking', stacking_clf,
    X_test_tree, y_test_arr,
    X_train_tree, y_train_final
)
results['Stacking'] = stacking_results

# ── Inspect meta-learner weights ──────────────────────────────────────────────
meta_lr   = stacking_clf.final_estimator_
meta_coefs = pd.DataFrame({
    'Base Model'               : ['Decision Tree', 'RF Tuned', 'XGBoost', 'LightGBM'],
    'Meta Weight (Churn class)': meta_lr.coef_[0]
}).sort_values('Meta Weight (Churn class)', ascending=False)

print("\nMeta-learner weights — how much it trusts each base model for churn:")
print(meta_coefs.to_string(index=False))
print("\nInterpretation: higher weight = meta-learner relies more on that model's probability")
print("when deciding if a customer will churn.")

# Bar chart of meta weights
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('white')
colors = ['#e94560' if v > 0 else '#4C72B0' for v in meta_coefs['Meta Weight (Churn class)']]
ax.barh(meta_coefs['Base Model'], meta_coefs['Meta Weight (Churn class)'],
        color=colors, edgecolor='white', height=0.5)
ax.axvline(0, color='#333', lw=1.2)
ax.set_xlabel('Meta-learner Coefficient (Churn class)', fontsize=11)
ax.set_title('Stacking — Meta-learner Trust per Base Model', fontweight='bold', fontsize=12)
ax.set_facecolor('#f8f9ff')
for i, v in enumerate(meta_coefs['Meta Weight (Churn class)']):
    ax.text(v + (0.02 if v>=0 else -0.02), i, f'{v:.3f}',
            va='center', ha='left' if v>=0 else 'right', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

<div style="background:#eef2ff; border-left:4px solid #4C72B0; padding:16px 24px; border-radius:0 8px 8px 0; margin:8px 0;">
  <h3 style="color:#0f3460; margin:0 0 8px 0;">15.4 — Threshold Optimisation</h3>
  <p style="color:#555; margin:0; line-height:1.6;">Default 0.5 ignores the 20:1 cost asymmetry (FN≈$200 vs FP≈$10). Sweeping 0.10–0.90, maximising F1, gives the data-driven operating point. The threshold is a <strong>business dial</strong> — adjustable without retraining.</p>
</div>

In [ ]:
# ── B4: Find best bonus model by AUC ─────────────────────────────────────────
bonus_names = ['LightGBM (Tuned)', 'Voting Ensemble', 'Stacking']
best_bonus_name = max(bonus_names, key=lambda k: results[k]['auc'])
best_bonus_model = results[best_bonus_name]['model']
print(f"Best bonus model by AUC: {best_bonus_name}  (AUC = {results[best_bonus_name]['auc']:.4f})")
print("Applying threshold optimisation on this model...")

In [ ]:
# ── B4: Threshold sweep ───────────────────────────────────────────────────────
y_proba_bonus = best_bonus_model.predict_proba(X_test_tree)[:, 1]

thresholds_b4   = np.arange(0.10, 0.90, 0.01)
f1_scores_b4    = []
recall_scores_b4 = []
prec_scores_b4  = []

for thresh in thresholds_b4:
    yp = (y_proba_bonus >= thresh).astype(int)
    f1_scores_b4.append(f1_score(y_test_arr, yp, zero_division=0))
    recall_scores_b4.append(recall_score(y_test_arr, yp, zero_division=0))
    prec_scores_b4.append(precision_score(y_test_arr, yp, zero_division=0))

# Optimal threshold for F1
optimal_idx    = np.argmax(f1_scores_b4)
optimal_thresh = thresholds_b4[optimal_idx]
optimal_f1     = f1_scores_b4[optimal_idx]
optimal_recall = recall_scores_b4[optimal_idx]
optimal_prec   = prec_scores_b4[optimal_idx]

default_f1 = f1_score(y_test_arr, (y_proba_bonus >= 0.5).astype(int))

print(f"Default threshold (0.50) → F1: {default_f1:.4f}")
print(f"Optimal threshold ({optimal_thresh:.2f}) → F1: {optimal_f1:.4f}  |  Recall: {optimal_recall:.4f}  |  Precision: {optimal_prec:.4f}")
print(f"F1 improvement: +{optimal_f1 - default_f1:.4f}")

# ── Plot threshold curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('white')

axes[0].plot(thresholds_b4, f1_scores_b4,    color='#4C72B0', lw=2.2, label='F1 Score')
axes[0].plot(thresholds_b4, recall_scores_b4, color='#e94560', lw=2.2, label='Recall')
axes[0].plot(thresholds_b4, prec_scores_b4,   color='#2ca02c', lw=2.2, label='Precision')
axes[0].axvline(optimal_thresh, color='#f5a623', ls='--', lw=2, label=f'Optimal = {optimal_thresh:.2f}')
axes[0].axvline(0.5, color='#888', ls=':', lw=1.5, label='Default = 0.50')
axes[0].scatter([optimal_thresh], [optimal_f1], color='#f5a623', s=120, zorder=5)
axes[0].set_xlabel('Decision Threshold', fontsize=11)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title(f'Threshold Optimisation — {best_bonus_name}\nF1, Recall, Precision curves',
                   fontweight='bold', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.05)
axes[0].set_facecolor('#f8f9ff')

# Precision-Recall tradeoff
axes[1].plot(recall_scores_b4, prec_scores_b4, color='#4C72B0', lw=2.2)
# Mark optimal point
axes[1].scatter([optimal_recall], [optimal_prec], color='#f5a623', s=150, zorder=5,
                label=f'Optimal t={optimal_thresh:.2f}')
axes[1].scatter([recall_scores_b4[list(thresholds_b4).index(min(thresholds_b4, key=lambda x: abs(x-0.5)))]],
                [prec_scores_b4[list(thresholds_b4).index(min(thresholds_b4, key=lambda x: abs(x-0.5)))]],
                color='#888', s=120, zorder=5, marker='D', label='Default t=0.50')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title(f'Precision-Recall Tradeoff\n({best_bonus_name})',
                   fontweight='bold', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].set_facecolor('#f8f9ff')

plt.suptitle('Threshold Optimisation Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Final prediction with optimal threshold ────────────────────────────────────
y_pred_optimal = (y_proba_bonus >= optimal_thresh).astype(int)
print("\nClassification report at optimal threshold:")
print(classification_report(y_test_arr, y_pred_optimal, target_names=['No Churn', 'Churn']))

# Store as bonus result
cm_opt = confusion_matrix(y_test_arr, y_pred_optimal)
tn_o, fp_o, fn_o, tp_o = cm_opt.ravel()
spec_opt = tn_o / (tn_o + fp_o) if (tn_o + fp_o) > 0 else 0.0
auc_opt  = roc_auc_score(y_test_arr, y_proba_bonus)
results['Best + Threshold'] = {
    'model': best_bonus_model, 'name': 'Best + Threshold',
    'accuracy': accuracy_score(y_test_arr, y_pred_optimal),
    'precision': optimal_prec, 'recall': optimal_recall,
    'specificity': spec_opt, 'f1': optimal_f1, 'auc': auc_opt,
    'train_f1': optimal_f1, 'gap': 0.0,
    'y_pred': y_pred_optimal, 'y_proba': y_proba_bonus
}
print(f"\nBest + Threshold added to results.")

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 16 &mdash; Full Model Evaluation &amp; Comparison</h2>
  <p style="color:#444; line-height:1.7;">All 11 models — baseline pipeline (Sections 12–14) and advanced ensembles (Section 15) — compared on a single unified table. The <code>results</code> dict accumulated results throughout. Bonus models highlighted for incremental improvement visibility.</p>
</div>

In [ ]:
# ── Unified comparison table — all 11 models ─────────────────────────────────
BONUS_NAMES = ['LightGBM (Tuned)', 'Voting Ensemble', 'Stacking', 'Best + Threshold']

comparison_rows = []
for name, res in results.items():
    comparison_rows.append({
        'Model'      : name,
        'Type'       : '▶ Ensemble' if name in BONUS_NAMES else 'Baseline',
        'Accuracy'   : round(res['accuracy'],   4),
        'Precision'  : round(res['precision'],  4),
        'Recall'     : round(res['recall'],     4),
        'Specificity': round(res['specificity'],4),
        'F1 Score'   : round(res['f1'],         4),
        'AUC-ROC'    : round(res['auc'],        4),
        'Gap'        : round(res['gap'],        4)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('F1 Score', ascending=False)
best_model_name = comparison_df.iloc[0]['Model']

print("COMPLETE MODEL COMPARISON — BASELINE + ENSEMBLES (sorted by F1 Score)")
print("=" * 100)
print(comparison_df.to_string(index=False))
print(f"\n★  Best model: {best_model_name}")
print(f"   Recall   : {comparison_df.iloc[0]['Recall']:.4f}")
print(f"   F1 Score : {comparison_df.iloc[0]['F1 Score']:.4f}")
print(f"   AUC-ROC  : {comparison_df.iloc[0]['AUC-ROC']:.4f}")

# Improvement summary
baseline_f1  = max(res['f1']  for n, res in results.items() if n not in BONUS_NAMES)
baseline_auc = max(res['auc'] for n, res in results.items() if n not in BONUS_NAMES)
print(f"\nBest baseline → F1={baseline_f1:.4f}  AUC={baseline_auc:.4f}")
print("Ensemble improvements:")
for _, row in comparison_df[comparison_df['Type']=='▶ Ensemble'].iterrows():
    df1 = row['F1 Score'] - baseline_f1
    da  = row['AUC-ROC']  - baseline_auc
    s = lambda v: f"+{v:.4f}" if v>=0 else f"{v:.4f}"
    print(f"  {row['Model']:28s} → F1 {s(df1)}  |  AUC {s(da)}")


In [ ]:
# ── ROC curves — all models, ensembles bold ──────────────────────────────────
from matplotlib.patches import Patch
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor('white')
palette_roc = ['#4C72B0','#dd8452','#55a868','#c44e52','#8172b3','#937860','#da8bc3']

for i, (name, res) in enumerate(results.items()):
    if res['y_proba'] is None: continue
    is_ens = name in BONUS_NAMES
    fpr, tpr, _ = roc_curve(y_test_arr, res['y_proba'])
    color = '#e94560' if name == 'Best + Threshold' else '#f5a623' if is_ens else palette_roc[i % 7]
    ax.plot(fpr, tpr, lw=2.8 if is_ens else 1.4, alpha=1.0 if is_ens else 0.5, color=color,
            label=f"{name} (AUC={res['auc']:.3f}){'  ◀' if name==best_model_name else ''}")

ax.plot([0,1],[0,1],'k--',lw=1.2,alpha=0.4,label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves — All Models  (bold lines = ensemble models)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=8.5)
ax.set_facecolor('#f8f9ff')
fig.legend(handles=[Patch(color='#e94560',label='Best deployed model'),
                    Patch(color='#f5a623',label='Other ensembles'),
                    Patch(color='#4C72B0',label='Baseline models')],
           loc='upper left', bbox_to_anchor=(0.12,0.88), fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# ── Unified bar chart — baseline vs ensemble ─────────────────────────────────
metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
fig, axes = plt.subplots(1, 5, figsize=(26, 6))
fig.patch.set_facecolor('white')

for i, metric in enumerate(metrics_plot):
    vals  = comparison_df[metric].values
    names = comparison_df['Model'].values
    types = comparison_df['Type'].values
    colors = ['#e94560' if (t=='▶ Ensemble' and v==max(vals)) else
              '#f5a623' if t=='▶ Ensemble' else '#4C72B0'
              for t, v in zip(types, vals)]
    axes[i].bar(range(len(names)), vals, color=colors, edgecolor='white')
    axes[i].set_xticks(range(len(names)))
    axes[i].set_xticklabels(names, rotation=38, ha='right', fontsize=7)
    axes[i].set_ylim(0, 1.14)
    axes[i].set_title(metric, fontweight='bold', fontsize=11)
    axes[i].set_facecolor('#f8f9ff')
    axes[i].yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
    for j, v in enumerate(vals):
        axes[i].text(j, v+0.01, f'{v:.3f}', ha='center', fontsize=6.5,
                     fontweight='bold' if types[j]=='▶ Ensemble' else 'normal',
                     color='#e94560' if (types[j]=='▶ Ensemble' and v==max(vals)) else '#333')

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color='#e94560',label='Best ensemble'),
                    Patch(color='#f5a623',label='Other ensembles'),
                    Patch(color='#4C72B0',label='Baseline')],
           loc='upper right', fontsize=10)
plt.suptitle('Complete Model Comparison — Baseline vs Ensembles', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 17 &mdash; Feature Importance</h2>
  <p style="color:#444; line-height:1.7;">Importances from the best tree-based model reveal which signals drive predictions most. Directly actionable: high-importance features point to root causes operations can address.</p>
</div>

In [ ]:
# ── Extract feature importance from best tree model ───────────────────────────
tree_models = {k: v for k, v in results.items()
               if hasattr(v['model'], 'feature_importances_')}

if tree_models:
    best_tree_name = max(tree_models, key=lambda k: tree_models[k]['f1'])
    best_tree = tree_models[best_tree_name]['model']
    feat_imp = pd.Series(best_tree.feature_importances_, index=top25).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(12, 9))
    palette = sns.color_palette('RdBu_r', n_colors=25)
    bars = ax.barh(range(25), feat_imp.values[::-1], color=palette, edgecolor='white')
    ax.set_yticks(range(25))
    ax.set_yticklabels(feat_imp.index[::-1], fontsize=9)
    ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
    ax.set_title(f'Feature Importances — {best_tree_name}', fontsize=13, fontweight='bold')

    # Value labels
    for i, (bar, val) in enumerate(zip(bars, feat_imp.values[::-1])):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=7)
    plt.tight_layout()
    plt.show()

    print(f"\nTop 10 most important features ({best_tree_name}):")
    print(feat_imp.head(10).to_string())
else:
    print("No tree-based model available for feature importance")
    feat_imp = None

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 18 &mdash; Business Dashboard</h2>
  <p style="color:#444; line-height:1.7;">Interactive Plotly charts for non-technical stakeholders. Risk segments classify customers into Retention Priority tiers. All charts use the best model — automatically benefits from the ensembles above. Interactive: hover, zoom, filter.</p>
</div>

In [ ]:
# ── Use best model for business dashboard ─────────────────────────────────────
best_result = max(results.values(), key=lambda r: r['f1'])
best_model_obj = best_result['model']
best_proba = best_result['y_proba']

if best_proba is None:
    # Fallback if no probability available
    best_proba = best_result['y_pred'].astype(float)

print(f"Dashboard using: {best_result['name']}")
print(f"Churn probability range: [{best_proba.min():.4f}, {best_proba.max():.4f}]")

In [ ]:
# ── 17.1 Churn probability histogram ─────────────────────────────────────────
fig = px.histogram(
    x=best_proba, color=y_test_arr.astype(str),
    nbins=50, barmode='overlay', opacity=0.7,
    color_discrete_map={'0': '#4C72B0', '1': '#e94560'},
    labels={'x': 'Predicted Churn Probability', 'color': 'Actual'},
    title=f'Churn Probability Distribution — {best_result["name"]}',
    template='plotly_white'
)
fig.update_layout(
    legend_title='Actual Churn',
    font=dict(family='Arial', size=12),
    title_font_size=15,
    plot_bgcolor='#f8f9ff'
)
fig.show()

In [ ]:
# ── 17.2 Risk segment pie chart ───────────────────────────────────────────────
risk_labels = pd.cut(best_proba,
                     bins=[0, 0.3, 0.6, 1.0],
                     labels=['Low Risk\n(0-30%)', 'Medium Risk\n(30-60%)', 'High Risk\n(60-100%)'])
risk_counts = risk_labels.value_counts()

fig = px.pie(
    names=risk_counts.index,
    values=risk_counts.values,
    title='Customer Risk Segmentation by Churn Probability',
    color_discrete_sequence=['#4C72B0', '#f5a623', '#e94560'],
    hole=0.4,
    template='plotly_white'
)
fig.update_traces(textposition='outside', textinfo='percent+label',
                  pull=[0, 0, 0.05])
fig.update_layout(font=dict(family='Arial', size=12), title_font_size=15)
fig.show()

print("\nRisk Segment Breakdown:")
for seg, cnt in risk_counts.items():
    print(f"  {str(seg):25s}: {cnt:,} customers ({cnt/len(best_proba)*100:.1f}%)")

In [ ]:
# ── 17.3 Interactive Feature Importance ───────────────────────────────────────
if feat_imp is not None:
    top15_fi = feat_imp.head(15)
    fig = px.bar(
        x=top15_fi.values, y=top15_fi.index,
        orientation='h',
        title=f'Top 15 Feature Importances — {best_tree_name}',
        labels={'x': 'Importance Score', 'y': 'Feature'},
        color=top15_fi.values,
        color_continuous_scale=['#4C72B0', '#e94560'],
        template='plotly_white'
    )
    fig.update_layout(
        yaxis={'categoryorder': 'total ascending'},
        coloraxis_showscale=False,
        font=dict(family='Arial', size=11),
        title_font_size=15,
        plot_bgcolor='#f8f9ff'
    )
    fig.show()

In [ ]:
# ── 17.4 Interactive ROC curve ────────────────────────────────────────────────
fig = go.Figure()
colors_interactive = px.colors.qualitative.Set2

for i, (name, res) in enumerate(results.items()):
    if res['y_proba'] is not None:
        fpr, tpr, thresholds = roc_curve(y_test_arr, res['y_proba'])
        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            name=f"{name} (AUC={res['auc']:.3f})",
            mode='lines', line=dict(width=2.5),
            hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'
        ))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                          name='Random', line=dict(dash='dash', color='grey', width=1.5)))
fig.update_layout(
    title='Interactive ROC Curves — All Models',
    xaxis_title='False Positive Rate (1 - Specificity)',
    yaxis_title='True Positive Rate (Recall)',
    template='plotly_white',
    font=dict(family='Arial', size=12),
    title_font_size=15,
    legend=dict(x=0.65, y=0.05),
    hovermode='x unified',
    plot_bgcolor='#f8f9ff'
)
fig.show()

In [ ]:
# ── 17.5 Interactive confusion matrix heatmap ─────────────────────────────────
best_cm = confusion_matrix(y_test_arr, best_result['y_pred'])
tn, fp, fn, tp = best_cm.ravel()

cm_labels = [['True Negative\n(Loyal - Correct)', 'False Positive\n(Loyal - Flagged)'],
             ['False Negative\n(Churner - Missed)', 'True Positive\n(Churner - Caught)']]
cm_values = [[tn, fp], [fn, tp]]

fig = go.Figure(data=go.Heatmap(
    z=cm_values,
    x=['Predicted: Not Churned', 'Predicted: Churned'],
    y=['Actual: Not Churned', 'Actual: Churned'],
    colorscale=[[0, '#f8f9ff'], [0.5, '#4C72B0'], [1, '#e94560']],
    showscale=True,
    text=[[f'{cm_labels[i][j]}<br><b>{cm_values[i][j]:,}</b>'
           for j in range(2)] for i in range(2)],
    texttemplate='%{text}',
    hovertemplate='Count: %{z}<extra></extra>'
))
fig.update_layout(
    title=f'Interactive Confusion Matrix — {best_result["name"]}',
    font=dict(family='Arial', size=12),
    title_font_size=15,
    template='plotly_white',
    plot_bgcolor='#f8f9ff'
)
fig.show()

print(f"\nBusiness Impact Summary:")
print(f"  Churners correctly identified (TP): {tp:,}")
print(f"  Churners missed          (FN): {fn:,}  <- Revenue at risk")
print(f"  False alarms             (FP): {fp:,}  <- Wasted retention spend")
print(f"  Loyal customers correct  (TN): {tn:,}")
print(f"  Churn capture rate (Recall): {tp/(tp+fn)*100:.1f}%")

<div style="background:#f8f9ff; border-left:5px solid #e94560; padding:24px 32px; border-radius:0 12px 12px 0; margin:12px 0;">
  <h2 style="color:#0f3460; margin-top:0;">Section 19 &mdash; Revenue at Risk Analysis</h2>
  <p style="color:#444; line-height:1.7;">Directly answers: <strong>how much revenue are we about to lose?</strong> Using the model's predicted churn probabilities on the test set, combined with each customer's revenue profile (<code>rev_Mean</code>, <code>totrev</code>, <code>adjrev</code>, <code>avg3rev</code>, <code>avg6rev</code>, <code>ovrrev_Mean</code>, <code>vceovr_Mean</code>, <code>datovr_Mean</code>, <code>totmrc_Mean</code>), we quantify exposure and prioritise the retention budget.</p>
</div>

In [ ]:
# ── Section 19: Revenue at Risk Analysis ─────────────────────────────────────
# Revenue columns available in the original dataset
REV_COLS = ['rev_Mean', 'totrev', 'adjrev', 'avgrev', 'avg3rev', 'avg6rev',
            'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'totmrc_Mean']

# ── 19.1 Reconstruct test-set revenue profile ─────────────────────────────────
# The test set index maps back to df_fe (which shares index with df_clean / df)
# We use X_test's index to retrieve the matching rows from df_clean
test_idx = X_test.index  # pandas index from the pre-scaling X_final

rev_available = [c for c in REV_COLS if c in df_clean.columns]
df_test_rev = df_clean.loc[test_idx, rev_available].copy()

# Attach model outputs
df_test_rev['churn_actual']   = y_test_arr
df_test_rev['churn_proba']    = results[best_model_name]['y_proba']
df_test_rev['churn_predicted']= results[best_model_name]['y_pred']

# Fallback: if rev_Mean missing, use avgrev or totmrc_Mean
if 'rev_Mean' not in df_test_rev.columns:
    if 'avgrev' in df_test_rev.columns:
        df_test_rev['rev_Mean'] = df_test_rev['avgrev']
    elif 'totmrc_Mean' in df_test_rev.columns:
        df_test_rev['rev_Mean'] = df_test_rev['totmrc_Mean']
    else:
        df_test_rev['rev_Mean'] = 50.0  # safe fallback
        print("WARNING: no revenue column found — using placeholder $50")

# Revenue at Risk = churn_proba × rev_Mean (expected monthly loss per customer)
df_test_rev['expected_monthly_loss'] = df_test_rev['churn_proba'] * df_test_rev['rev_Mean']

# Lifetime revenue at risk (for predicted churners only)
if 'totrev' in df_test_rev.columns:
    df_test_rev['lifetime_at_risk'] = df_test_rev['churn_proba'] * df_test_rev['totrev']

print("Revenue columns available in test set:", rev_available)
print(f"Test set customers: {len(df_test_rev):,}")
print(f"Predicted churners: {df_test_rev['churn_predicted'].sum():,}")
print()
print(f"  rev_Mean range   : ${df_test_rev['rev_Mean'].min():.2f} – ${df_test_rev['rev_Mean'].max():.2f}")
print(f"  rev_Mean median  : ${df_test_rev['rev_Mean'].median():.2f}")
if 'totrev' in df_test_rev.columns:
    print(f"  totrev median    : ${df_test_rev['totrev'].median():.2f}")


In [ ]:
# ── 19.2 Headline exposure numbers ───────────────────────────────────────────
predicted_churners = df_test_rev[df_test_rev['churn_predicted'] == 1]

total_monthly_at_risk  = predicted_churners['rev_Mean'].sum()
total_expected_loss    = df_test_rev['expected_monthly_loss'].sum()  # probability-weighted
avg_rev_churner        = predicted_churners['rev_Mean'].mean()
n_predicted_churners   = len(predicted_churners)

# Missed churners (FN) — the invisible leak
fn_mask = (df_test_rev['churn_actual']==1) & (df_test_rev['churn_predicted']==0)
missed_monthly_rev = df_test_rev.loc[fn_mask, 'rev_Mean'].sum()

# Annualise
annual_risk = total_monthly_at_risk * 12
annual_expected = total_expected_loss * 12

print("=" * 62)
print("  REVENUE AT RISK — HEADLINE NUMBERS")
print("=" * 62)
print(f"  Predicted churners         : {n_predicted_churners:,} customers")
print(f"  Avg monthly rev / churner  : ${avg_rev_churner:,.2f}")
print()
print(f"  Monthly revenue at risk    : ${total_monthly_at_risk:>12,.2f}")
print(f"  Annual revenue at risk     : ${annual_risk:>12,.2f}")
print()
print(f"  Probability-weighted loss  : ${total_expected_loss:>12,.2f}  / month")
print(f"  Probability-weighted loss  : ${annual_expected:>12,.2f}  / year")
print()
print(f"  Missed churners (FN)       : {fn_mask.sum():,} customers")
print(f"  Undetected monthly leak    : ${missed_monthly_rev:>12,.2f}")
print(f"  Undetected annual leak     : ${missed_monthly_rev*12:>12,.2f}")
print()
print(f"  Churn capture rate         : {(1 - fn_mask.sum()/(df_test_rev['churn_actual'].sum()))*100:.1f}%")
print("=" * 62)


In [ ]:
# ── 19.3 Revenue tier segmentation ───────────────────────────────────────────
# Segment predicted churners into revenue tiers to prioritise retention budget

df_churners = predicted_churners.copy()
df_churners['rev_tier'] = pd.qcut(
    df_churners['rev_Mean'],
    q=4,
    labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']
)

tier_summary = df_churners.groupby('rev_tier', observed=True).agg(
    customers     = ('rev_Mean', 'count'),
    total_rev     = ('rev_Mean', 'sum'),
    avg_rev       = ('rev_Mean', 'mean'),
    avg_proba     = ('churn_proba', 'mean'),
    expected_loss = ('expected_monthly_loss', 'sum')
).reset_index()

tier_summary['pct_of_exposure'] = tier_summary['total_rev'] / tier_summary['total_rev'].sum() * 100
tier_summary['roi_priority'] = (tier_summary['expected_loss'] / tier_summary['customers']).round(2)

print("REVENUE TIER BREAKDOWN — PREDICTED CHURNERS")
print("=" * 82)
print(tier_summary[['rev_tier','customers','avg_rev','avg_proba','total_rev','pct_of_exposure','roi_priority']].to_string(index=False))
print()
print("roi_priority = expected monthly loss per customer in that tier")
print("→ Spend retention budget proportionally to roi_priority, not customer count")

# ── Visual ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('white')
palette_tiers = ['#4C72B0', '#55a868', '#f5a623', '#e94560']

# Bar: customers per tier
axes[0].bar(tier_summary['rev_tier'], tier_summary['customers'],
            color=palette_tiers, edgecolor='white')
axes[0].set_title('Predicted Churners by Revenue Tier', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Number of Customers')
axes[0].set_facecolor('#f8f9ff')
for i, v in enumerate(tier_summary['customers']):
    axes[0].text(i, v+5, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

# Bar: total monthly revenue at risk
axes[1].bar(tier_summary['rev_tier'], tier_summary['total_rev'],
            color=palette_tiers, edgecolor='white')
axes[1].set_title('Monthly Revenue at Risk by Tier', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Monthly Revenue ($)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_facecolor('#f8f9ff')
for i, v in enumerate(tier_summary['total_rev']):
    axes[1].text(i, v+100, f'${v:,.0f}', ha='center', fontsize=8, fontweight='bold')

# Horizontal bar: % of total exposure
axes[2].barh(tier_summary['rev_tier'], tier_summary['pct_of_exposure'],
             color=palette_tiers[::-1], edgecolor='white')
axes[2].set_title('Share of Total Revenue Exposure', fontweight='bold', fontsize=12)
axes[2].set_xlabel('% of Total Monthly Revenue at Risk')
axes[2].set_facecolor('#f8f9ff')
for i, v in enumerate(tier_summary['pct_of_exposure']):
    axes[2].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.suptitle('Revenue Tier Analysis — Where to Focus Retention Spend', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── 19.4 Early-warning: revenue declining customers (pre-churn signal) ─────────
# Customers where avg3rev < avg6rev — revenue is already falling BEFORE they churn
# These are the highest-priority cohort: the model predicts churn AND revenue is declining

early_warning_mask = None
if 'avg3rev' in df_test_rev.columns and 'avg6rev' in df_test_rev.columns:
    df_test_rev['rev_declining'] = (df_test_rev['avg3rev'] < df_test_rev['avg6rev']).astype(int)
    df_test_rev['rev_decline_pct'] = (
        (df_test_rev['avg6rev'] - df_test_rev['avg3rev']) / (df_test_rev['avg6rev'].abs() + 1e-6) * 100
    ).clip(0, 100)

    early_warning = df_test_rev[
        (df_test_rev['churn_predicted'] == 1) &
        (df_test_rev['rev_declining'] == 1)
    ].copy()

    print("EARLY-WARNING COHORT: Predicted churners with declining revenue")
    print("=" * 62)
    print(f"  Customers flagged      : {len(early_warning):,}")
    pct = len(early_warning)/max(1, df_test_rev['churn_predicted'].sum())*100
    print(f"  % of predicted churners: {pct:.1f}%")
    print(f"  Monthly rev at risk    : ${early_warning['rev_Mean'].sum():,.2f}")
    print(f"  Avg revenue decline    : {early_warning['rev_decline_pct'].mean():.1f}%")
    print(f"  Avg churn probability  : {early_warning['churn_proba'].mean():.3f}")
    print()
    print("  → These customers show BOTH ML-predicted churn AND observable revenue decline.")
    print("  → Highest ROI for proactive outreach: the signal arrived before the event.")

    # Distribution of decline %
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor('white')

    axes[0].hist(early_warning['rev_decline_pct'], bins=30, color='#e94560', alpha=0.8, edgecolor='white')
    axes[0].set_title('Revenue Decline % — Early Warning Cohort\n(predicted churners)', fontweight='bold')
    axes[0].set_xlabel('Revenue Decline % (avg6rev → avg3rev)')
    axes[0].set_ylabel('Number of Customers')
    axes[0].set_facecolor('#f8f9ff')

    axes[1].scatter(early_warning['rev_decline_pct'], early_warning['churn_proba'],
                    alpha=0.3, color='#e94560', s=15)
    axes[1].set_xlabel('Revenue Decline %')
    axes[1].set_ylabel('Churn Probability')
    axes[1].set_title('Revenue Decline vs Churn Probability\n(early-warning cohort)', fontweight='bold')
    axes[1].set_facecolor('#f8f9ff')
    axes[1].axhline(0.5, color='#333', ls='--', lw=1, alpha=0.5, label='Default threshold')
    axes[1].legend(fontsize=9)

    plt.suptitle('Early-Warning Cohort: Revenue Decline + Predicted Churn', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print("avg3rev / avg6rev not available in test set — skipping trend analysis.")
    print("Revenue trend signal (rev_trending_down) was used in feature engineering instead.")


In [ ]:
# ── 19.5 Overage risk cohort ─────────────────────────────────────────────────
# Customers paying overage are both high-revenue AND high-churn-risk
# ovrrev_Mean: mean monthly overage revenue (vceovr + datovr components)
# These are customers being charged extra — the most urgent retention target

ovr_cols_avail = [c for c in ['ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean'] if c in df_test_rev.columns]

if ovr_cols_avail:
    df_test_rev['total_overage_rev'] = df_test_rev[ovr_cols_avail].sum(axis=1)
    df_test_rev['is_overage_customer'] = (df_test_rev['total_overage_rev'] > 0).astype(int)

    ovr_churners = df_test_rev[
        (df_test_rev['churn_predicted'] == 1) &
        (df_test_rev['is_overage_customer'] == 1)
    ]
    non_ovr_churners = df_test_rev[
        (df_test_rev['churn_predicted'] == 1) &
        (df_test_rev['is_overage_customer'] == 0)
    ]

    print("OVERAGE RISK COHORT — Predicted churners paying overage fees")
    print("=" * 62)
    print(f"  Overage churners       : {len(ovr_churners):,}")
    print(f"  Non-overage churners   : {len(non_ovr_churners):,}")
    print(f"  Avg monthly rev (overage): ${ovr_churners['rev_Mean'].mean():,.2f}")
    print(f"  Avg monthly rev (non-ovr): ${non_ovr_churners['rev_Mean'].mean():,.2f}")
    print(f"  Monthly rev at risk (overage): ${ovr_churners['rev_Mean'].sum():,.2f}")
    print(f"  Avg churn prob (overage)  : {ovr_churners['churn_proba'].mean():.3f}")
    print(f"  Avg churn prob (non-ovr)  : {non_ovr_churners['churn_proba'].mean():.3f}")
    print()
    print("  → Overage customers: higher revenue AND higher churn probability.")
    print("  → Business action: auto plan-upgrade alert BEFORE overage is charged.")
    print("  → Overage revenue would be retained — net neutral to the business.")

    # ── Overage breakdown chart ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor('white')

    # Revenue distribution: overage vs non-overage churners
    axes[0].hist(ovr_churners['rev_Mean'], bins=40, alpha=0.75,
                 color='#e94560', label=f'Overage ({len(ovr_churners):,})', edgecolor='white')
    axes[0].hist(non_ovr_churners['rev_Mean'], bins=40, alpha=0.55,
                 color='#4C72B0', label=f'No Overage ({len(non_ovr_churners):,})', edgecolor='white')
    axes[0].set_title('Monthly Revenue Distribution\nOverage vs Non-Overage Churners', fontweight='bold')
    axes[0].set_xlabel('Monthly Revenue ($)')
    axes[0].set_ylabel('Customers')
    axes[0].legend(fontsize=9)
    axes[0].set_facecolor('#f8f9ff')

    # Overage component breakdown (stacked bar)
    ovr_totals = {c: df_test_rev.loc[ovr_churners.index, c].sum()
                  for c in ovr_cols_avail}
    axes[1].bar(list(ovr_totals.keys()), list(ovr_totals.values()),
                color=['#e94560', '#f5a623', '#4C72B0'][:len(ovr_totals)], edgecolor='white')
    axes[1].set_title('Overage Revenue at Risk by Component\n(predicted churners with overage)',
                       fontweight='bold')
    axes[1].set_ylabel('Total Monthly Overage Revenue ($)')
    axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[1].set_facecolor('#f8f9ff')
    for i, (k, v) in enumerate(ovr_totals.items()):
        axes[1].text(i, v+10, f'${v:,.0f}', ha='center', fontsize=10, fontweight='bold')

    plt.suptitle('Overage Risk Cohort Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print("Overage revenue columns not available in test set.")
    print("Use overage_ratio from feature engineering as a proxy.")


In [ ]:
# ── 19.6 Revenue at Risk — interactive Plotly summary ─────────────────────────
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Churn Probability vs Monthly Revenue',
        'Expected Monthly Loss Distribution',
        'Revenue Tier — Customers & Exposure',
        'Retention ROI by Revenue Tier'
    )
)

# Scatter: probability vs revenue
fig.add_trace(go.Scatter(
    x=df_test_rev['churn_proba'],
    y=df_test_rev['rev_Mean'],
    mode='markers',
    marker=dict(
        color=df_test_rev['churn_predicted'],
        colorscale=[[0,'#4C72B0'],[1,'#e94560']],
        opacity=0.4, size=4
    ),
    name='Customers',
    hovertemplate='Prob: %{x:.2f}<br>Rev: $%{y:.2f}<extra></extra>'
), row=1, col=1)
fig.add_vline(x=0.5, line_dash='dash', line_color='grey', row=1, col=1)

# Histogram: expected monthly loss
fig.add_trace(go.Histogram(
    x=df_test_rev[df_test_rev['churn_predicted']==1]['expected_monthly_loss'],
    nbinsx=40, marker_color='#e94560', opacity=0.75, name='Expected Loss',
    hovertemplate='Loss: $%{x:.2f}<br>Count: %{y}<extra></extra>'
), row=1, col=2)

# Bar: tier customers
for tier, clr in zip(tier_summary['rev_tier'], ['#4C72B0','#55a868','#f5a623','#e94560']):
    row_t = tier_summary[tier_summary['rev_tier']==tier].iloc[0]
    fig.add_trace(go.Bar(
        x=[str(tier)], y=[row_t['customers']], name=str(tier),
        marker_color=clr, showlegend=False,
        hovertemplate=f'{tier}<br>Customers: {row_t["customers"]:,}<extra></extra>'
    ), row=2, col=1)

# Bar: roi priority per tier
for tier, clr in zip(tier_summary['rev_tier'], ['#4C72B0','#55a868','#f5a623','#e94560']):
    row_t = tier_summary[tier_summary['rev_tier']==tier].iloc[0]
    fig.add_trace(go.Bar(
        x=[str(tier)], y=[row_t['roi_priority']], name=str(tier),
        marker_color=clr, showlegend=False,
        hovertemplate=f'{tier}<br>ROI Priority: ${row_t["roi_priority"]:.2f}/customer<extra></extra>'
    ), row=2, col=2)

fig.update_layout(
    height=700,
    title_text=f'Revenue at Risk Dashboard — {best_model_name}',
    title_font_size=16,
    template='plotly_white',
    font=dict(family='Arial', size=11)
)
fig.update_xaxes(title_text='Churn Probability', row=1, col=1)
fig.update_yaxes(title_text='Monthly Revenue ($)', row=1, col=1)
fig.update_xaxes(title_text='Expected Monthly Loss ($)', row=1, col=2)
fig.update_yaxes(title_text='Customers', row=1, col=2)
fig.update_xaxes(title_text='Revenue Tier', row=2, col=1)
fig.update_yaxes(title_text='Customers', row=2, col=1)
fig.update_xaxes(title_text='Revenue Tier', row=2, col=2)
fig.update_yaxes(title_text='ROI Priority ($/customer)', row=2, col=2)
fig.show()

print()
print("REVENUE AT RISK — EXECUTIVE SUMMARY")
print("=" * 62)
total_rar = predicted_churners['rev_Mean'].sum()
expected_rar = df_test_rev['expected_monthly_loss'].sum()
print(f"  Total monthly revenue at risk      : ${total_rar:>12,.2f}")
print(f"  Probability-weighted expected loss : ${expected_rar:>12,.2f}")
print(f"  Annualised exposure                : ${total_rar*12:>12,.2f}")
print(f"  Undetected annual leak (FN)        : ${missed_monthly_rev*12:>12,.2f}")
top_tier = tier_summary.sort_values('roi_priority', ascending=False).iloc[0]
print(f"  Highest ROI retention tier         : {top_tier['rev_tier']}  (${top_tier['roi_priority']:.2f}/customer/month)")
print()
print("  Recommendation: target High (Q4) tier first — highest expected return")
print("  per retention offer. Early-warning cohort (declining revenue) should")
print("  be contacted within 14 days of first detected revenue drop.")


<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%); padding:48px 40px; border-radius:18px; margin:24px 0;">
  <h1 style="color:#e94560; margin-top:0; font-size:2em; letter-spacing:1px; text-align:center; text-transform:uppercase;">
    Section 20 — Conclusion &amp; Action Plan
  </h1>
  <p style="color:#a8b2d8; text-align:center; font-size:1.05em; margin-bottom:32px;">
    100,000 Customers · 100 Features · 11 Models · Revenue at Risk Quantified
  </p>
  <h2 style="color:#a8b2d8; border-top:1px solid #e9456033; padding-top:20px; margin-top:28px;">1 · Pipeline Summary</h2>
  <p style="color:#c8d0e7; line-height:1.9;">
    <strong style="color:#e94560;">Phase 1 (Sections 1–11):</strong> 100 raw features → EDA → cleaning → 8 engineered signals → 4-stage selection funnel → 25 features → SMOTE + scaling.<br>
    <strong style="color:#e94560;">Phase 2 (Sections 12–14):</strong> 5 baseline classifiers with hyperparameter tuning.<br>
    <strong style="color:#e94560;">Phase 3 (Section 15):</strong> LightGBM + Soft Voting + Stacking + threshold calibration.<br>
    <strong style="color:#e94560;">Phase 4 (Sections 16–19):</strong> Unified evaluation, feature importance, business dashboard, revenue at risk quantification.
  </p>

  <h2 style="color:#a8b2d8; border-top:1px solid #e9456033; padding-top:20px; margin-top:28px;">2 · Model Progression</h2>
  <ul style="color:#c8d0e7; line-height:1.9;">
    <li><strong>Logistic Regression / KNN:</strong> Transparent baselines. Limited recall on the minority class — the floor.</li>
    <li><strong>Decision Tree:</strong> Interpretable rules for frontline teams. Depth-constrained to generalise.</li>
    <li><strong>Random Forest (Tuned):</strong> Solid F1 via bagging. Stable anchor for ensembles.</li>
    <li><strong>XGBoost (Tuned):</strong> Best single model — captures non-linear usage interactions.</li>
    <li><strong>LightGBM (Tuned):</strong> Faster on wide data, comparable accuracy. Leaf-wise growth finds finer splits.</li>
    <li><strong>Soft Voting:</strong> Probability averaging cancels structural errors across three booster types.</li>
    <li><strong>Stacking:</strong> Meta-learner learns which model to trust per customer. Highest discrimination.</li>
    <li><strong style="color:#e94560;">Best + Threshold (deploy this):</strong> Highest-AUC ensemble + data-driven threshold. Adjust Recall without retraining.</li>
  </ul>

  <h2 style="color:#a8b2d8; border-top:1px solid #e9456033; padding-top:20px; margin-top:28px;">3 · Top Churn Signals</h2>
  <ul style="color:#c8d0e7; line-height:1.9;">
    <li><strong style="color:#e94560;">Revenue decline (rev_trending_down, change_rev):</strong> Earliest signal — outreach within 14 days of first drop.</li>
    <li><strong style="color:#e94560;">High customer care calls:</strong> Frequent complaints = unresolved frustration. Dedicated callback programme.</li>
    <li><strong style="color:#e94560;">Low call completion rate:</strong> Network quality issue. Prioritise infrastructure in high drop-block areas.</li>
    <li><strong style="color:#e94560;">High overage ratio:</strong> Being charged extra = primed to leave. Auto plan-upgrade alerts before overage triggers.</li>
    <li><strong style="color:#e94560;">Short tenure (0–6 months):</strong> Highest churn risk window. Onboarding programme + early loyalty rewards.</li>
    <li><strong style="color:#e94560;">Equipment age (2y+):</strong> Old device → churn risk. Subsidised upgrade offers.</li>
  </ul>

  <h2 style="color:#a8b2d8; border-top:1px solid #e9456033; padding-top:20px; margin-top:28px;">4 · Revenue at Risk Action Plan</h2>
  <ol style="color:#c8d0e7; line-height:1.9;">
    <li>Score all active customers <strong>monthly</strong> with Best + Threshold model. Flag High Risk tier.</li>
    <li>Prioritise <strong>Q4 revenue tier</strong> (highest ROI per retention offer). Allocate budget proportionally to <code>roi_priority</code>.</li>
    <li>Contact <strong>early-warning cohort</strong> (declining avg3rev vs avg6rev) within 14 days — highest ROI, event hasn't happened yet.</li>
    <li>Offer <strong>overage customers</strong> automatic plan upgrades — stops overage friction and retains their elevated ARPU.</li>
    <li>A/B test campaigns: personalised plan upgrade vs. discount vs. service credit. Track conversion monthly.</li>
  </ol>

  <h2 style="color:#a8b2d8; border-top:1px solid #e9456033; padding-top:20px; margin-top:28px;">5 · Monitoring &amp; Maintenance</h2>
  <ul style="color:#c8d0e7; line-height:1.8;">
    <li><strong>Primary KPI:</strong> Recall ≥ 0.70 in production. Retrain on latest 3 months if it drops.</li>
    <li><strong>Drift alert:</strong> AUC &lt; 0.70 on rolling validation window → retrain.</li>
    <li><strong>Threshold review:</strong> Quarterly — re-run sweep if cost ratio changes (retention offer cost, customer ARPU).</li>
    <li><strong>Leakage controls:</strong> SMOTE, scaler, and stacking CV all applied on training data only. Reproducible at <code>SEED=42</code>.</li>
  </ul>
</div>

In [ ]:
# ── Final executive summary print ────────────────────────────────────────────
print("=" * 70)
print("TELECOM CHURN PREDICTION — FINAL EXECUTIVE SUMMARY")
print("=" * 70)
print()
print("MODEL LEADERBOARD (sorted by F1 Score)")
print("-" * 70)
print(comparison_df[['Model','Type','Recall','F1 Score','AUC-ROC','Gap']].to_string(index=False))
print()
best_row = comparison_df.iloc[0]
print(f"★  Deploy           : {best_row['Model']}")
print(f"   Recall           : {best_row['Recall']:.4f}")
print(f"   F1 Score         : {best_row['F1 Score']:.4f}")
print(f"   AUC-ROC          : {best_row['AUC-ROC']:.4f}")
print()
print("REVENUE AT RISK (test set — scale to full customer base)")
print("-" * 70)
print(f"   Predicted churners          : {len(predicted_churners):,}")
print(f"   Monthly revenue at risk     : ${predicted_churners['rev_Mean'].sum():>12,.2f}")
print(f"   Annual revenue at risk      : ${predicted_churners['rev_Mean'].sum()*12:>12,.2f}")
print(f"   Probability-weighted loss   : ${df_test_rev['expected_monthly_loss'].sum():>12,.2f} / month")
print(f"   Undetected annual leak (FN) : ${missed_monthly_rev*12:>12,.2f}")
print()
print("Pipeline complete.")
